# 🕵️ Chapter 1: The Detective's Hypothesis

Before we train a single model, we must ask a dangerous question: **"Is the game rigged?"**

In Data Science, this is called **Distribution Drift**.
* **Imagine this:** You study for an exam using only *History* books (Training Data).
* **The Exam:** The teacher hands you a *Math* test (Test Data).
* **The Result:** You fail, even though you studied hard.

### 🧪 The Experiment
We will mix the `Train` and `Test` data together and ask a model to guess which is which.
* If the model relies on random guessing (AUC ~0.50), the data is safe.
* If the model can easily tell them apart (AUC > 0.60), we have **Drift**. The "History" books won't help us pass the "Math" test. 

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder


# Load the data
df_train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
df_test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s5e12/sample_submission.csv')

print(f"Train Shape: {df_train.shape}")
print(f"Test Shape: {df_test.shape}")
print(f"submission: {submission.shape}")

Train Shape: (700000, 26)
Test Shape: (300000, 25)
submission: (300000, 2)


In [4]:
# --- THE DRIFT CHECK ---

# 1. We start by creating a "Target" that isn't Diabetes.
# We want to see if we can predict "Is this row from the Test Set?"
df_train['is_test'] = 0   # Label Training rows as 0
df_test['is_test'] = 1    # Label Test rows as 1

# 2. We combine them into one giant dataset to confuse the model
df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# 3. The "Translator" (Label Encoding)
# Models only speak numbers. They don't understand "Male" or "Female".
# We find all text columns (object) and translate them into numbers (0, 1).
object_cols = df_all.select_dtypes(include=['object']).columns
for col in object_cols:
    le = LabelEncoder()
    df_all[col] = le.fit_transform(df_all[col].astype(str))

# 4. The Setup
# We drop 'diagnosed_diabetes' because that's the answer to the real test.
# We only want to look at the clues (features) to see if they look different.
X_drift = df_all.drop(['diagnosed_diabetes', 'is_test', 'id'], axis=1)
y_drift = df_all['is_test']

# 5. The Investigation (Training the Model)
print("🕵️ Detective Model is running...")
skf_drift = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
drift_scores = []

for fold, (train_idx, val_idx) in enumerate(skf_drift.split(X_drift, y_drift)):
    # Split the data
    X_tr, X_val = X_drift.iloc[train_idx], X_drift.iloc[val_idx]
    y_tr, y_val = y_drift.iloc[train_idx], y_drift.iloc[val_idx]
    
    # Ask XGBoost: "Can you tell the difference between Train and Test?"
    clf = xgb.XGBClassifier(eval_metric='logloss', max_depth=4, n_estimators=100)
    clf.fit(X_tr, y_tr)
    
    # Score the Detective
    pred = clf.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, pred)
    drift_scores.append(score)

print(f"🔎 Final Verdict (Drift AUC): {np.mean(drift_scores):.4f}")

🕵️ Detective Model is running...
🔎 Final Verdict (Drift AUC): 0.6291


# 🏗️ Chapter 2: The Honest Mirror

Now that we know the data is tricky (Drift AUC was ~0.63), we cannot trust the Public Leaderboard. Why? Because the Leaderboard only shows us **20%** of the test data. It is a "Sample Size Trap."

### 🛡️ The Strategy: Stratified K-Fold
To stop ourselves from hallucinating a good score, we use **Stratified K-Fold**.
* **"K-Fold":** We split our training data into 5 parts. We train on 4 and test on 1. We do this 5 times.
* **"Stratified":** This is the secret sauce. It ensures that if 10% of people have diabetes in the real world, exactly 10% have diabetes in every single fold.

**The Golden Rule:** We calculate the **OOF (Out-Of-Fold)** score. This is the score of our model predicting data it has *never seen before*. It is the only honest metric we have.

In [6]:
# --- THE BASELINE MODEL ---

# 1. Setup Data & Clean Slate
# We reload the data to ensure no "drift" variables from the previous chapter interfere.
df_train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
df_test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s5e12/sample_submission.csv')

# 2. The "Translator" (Label Encoding)
# Problem: XGBoost only speaks numbers. It doesn't understand "Male" or "High School".
# Solution: We verify all categorical columns and translate them into numbers (0, 1, 2...).
# Crucial: We must use the SAME translator for both Train and Test to avoid confusion.
object_cols = df_train.select_dtypes(include=['object']).columns

for col in object_cols:
    le = LabelEncoder()
    # We combine both datasets to learn ALL possible words before translating
    combined_data = pd.concat([df_train[col], df_test[col]], axis=0).astype(str)
    le.fit(combined_data)
    
    df_train[col] = le.transform(df_train[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))

# 3. The Setup
X = df_train.drop(['diagnosed_diabetes', 'id'], axis=1)
y = df_train['diagnosed_diabetes']
X_test = df_test.drop(['id'], axis=1)

# 4. The "Honest" Validation Scheme
# We use 5 folds. We lock the random_state (42) so this experiment is reproducible.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))       # To store the "Honest" predictions
test_preds = np.zeros(len(X_test)) # To store the submission predictions
scores = []

print("🏗️ Building the Baseline...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # A. The Split: "Study Material" vs "Mock Exam"
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # B. The Training
    # Using standard XGBoost hyperparameters. No fancy tuning yet.
    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        objective='binary:logistic',
        eval_metric='auc',
        early_stopping_rounds=50,
        n_jobs=-1, 
        random_state=42
    )
    
    model.fit(
        X_tr, y_tr, 
        eval_set=[(X_val, y_val)], 
        verbose=False
    )
    
    # C. The "Honest" Prediction (OOF)
    # We predict the fold we DID NOT train on.
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    
    # D. The "Future" Prediction (Test)
    # We average the predictions from all 5 folds to reduce variance.
    test_preds += model.predict_proba(X_test)[:, 1] / 5
    
    # Score this fold
    auc = roc_auc_score(y_val, val_pred)
    scores.append(auc)
    print(f"   ---> Fold {fold+1} Score: {auc:.5f}")

# 5. Final Report
mean_score = np.mean(scores)
std_score = np.std(scores)
print(f"\n✅ Model: XGBoost Baseline")
print(f"   CV Score (Mean): {mean_score:.5f}")
print(f"   Stability (Std): {std_score:.5f}")

# 6. Save the Evidence
submission['diagnosed_diabetes'] = test_preds
submission.to_csv('submission_exp001.csv', index=False)
oof_df = pd.DataFrame({'id': df_train['id'], 'pred': oof_preds, 'target': y})
oof_df.to_csv('oof_exp001.csv', index=False)
print("   Files saved successfully (OOF & Submission).")

🏗️ Building the Baseline...
   ---> Fold 1 Score: 0.72634
   ---> Fold 2 Score: 0.72446
   ---> Fold 3 Score: 0.72525
   ---> Fold 4 Score: 0.72676
   ---> Fold 5 Score: 0.72557

✅ Model: XGBoost Baseline
   CV Score (Mean): 0.72568
   Stability (Std): 0.00081
   Files saved successfully (OOF & Submission).


# 📉 Chapter 3: The Gap Analysis & Conclusion

We have finished our first experiment. Now comes the most important part of the Grandmaster Strategy: **Analyzing the Gap.**

We compare our internal "Honest Mirror" (CV) against the external "Reality Check" (Public LB).

### 📊 The Scorecard

| Metric | Score | Meaning |
| :--- | :--- | :--- |
| **1. CV Score** | `0.72568` | How well the model learns the **Training Data**. |
| **2. LB Score** | `0.69575` | How well the model generalizes to the **Test Data**. |
| **3. The Gap** | `+0.0299` | **DANGER.** The measure of Overfitting/Drift. |

---

### 🧠  What do these numbers actually mean?

#### **1. Why is the CV Score (0.72) so high?**
* **The Reason:** Our model found patterns in the Training data (e.g., *"People with High Cholesterol always have Diabetes"*). Inside the training set, this rule works perfectly.
* **The Effect:** This gives us a sense of false confidence. If we only looked at this number, we would think we are doing great.

#### **2. Why is the LB Score (0.69) lower?**
* **The Reason:** The "Rules" changed! In the Test set, maybe "High Cholesterol" doesn't always equal Diabetes anymore. The distribution of patients has shifted (Drift).
* **The Effect:** Our model is being punished for "memorizing" the Training data too closely.

#### **3. The "Gap" (+0.03): The Enemy**
* **The Concept:** This number measures our **delusion**.
* **The Goal:** A perfect model has a Gap of nearly 0.
* **The Reality:** A gap of 0.03 means our model is **Overfitting to the Training Distribution**. It is like studying for a History exam but taking a Geography test.

---

### 🖼️ The Visual: "The Moving Target"

Imagine you are an archer.
* **The Training Set** is a target standing 10 meters away. You hit the Bullseye every time (Score 0.72).
* **The Test Set** is a target that has been moved to 15 meters away and slightly to the left.
* **The Result:** You are still aiming at the 10-meter spot, so your arrow misses the Bullseye on the real target (Score 0.69).

```text
       [Training World]                 [Real World]
       
      🎯 Target is Here              💨 WIND (Drift) 💨
             |                                |
    🏹 You Aim Here (0.72)          🎯 Target Moved Here!
                                              |
                                     🏹 You still Aim Here (0.69)
                                              |
                                         ❌ MISS

### 📬 A Note on the Strategy

Thank you for checking out **Day 1** of this log!

Even as experienced Kagglers, we often skip these foundational steps (like Drift Checks) in favor of training models quickly. My goal with this series is to slow down and show **why** the fundamentals matter, especially in Playground competitions with synthetic data.

**My Plan:**
I will update this notebook daily/weekly as I tackle the specific challenges of S5E12:

* **Handling the drift in features**
* **Feature Selection to fix the Drift.**

I'm curious—how are you handling the drift in features like `Cholesterol`? Are you dropping them or binning them? Let's discuss in the comments! 👇